# 02 — Reduction Kernels

Covers Phase 2 of the kernel roadmap:
- `reduce_sum` — parallel tree reduction
- `max_min` — argmax, argmin
- `softmax` — row-wise, numerically stable
- `layer_norm` — forward pass

**Metric**: GB/s for most; softmax/layer_norm use `2 × n × bytes` (read once + write once).

In [ ]:
# ── Setup: mount Drive and clone / pull the repo ─────────────────────────────
import os
from google.colab import drive

drive.mount("/content/drive")

REPO_URL    = "https://github.com/Bhavikupadhyay/triton-kernels.git"
REPO_BRANCH = "feature/reduce-sum"
REPO_DIR    = "/content/drive/MyDrive/triton-kernels"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all
    !git -C {REPO_DIR} checkout -f {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!bash scripts/setup_colab.sh

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import triton

from kernels.reductions.reduce_sum import reduce_sum, test_reduce_sum, benchmark_reduce_sum

# Uncomment as kernels are implemented:
# from kernels.reductions.max_min import argmax, argmin, test_max_min, benchmark_max_min
# from kernels.reductions.softmax import softmax, test_softmax, benchmark_softmax
# from kernels.reductions.layer_norm import layer_norm, test_layer_norm, benchmark_layer_norm

print("Imports ready")

## 1. reduce_sum

**File**: `kernels/reductions/reduce_sum.py`  
**PyTorch equivalent**: `torch.sum(x)`  
**Design**: two-pass — each block reduces BLOCK_SIZE elements into one partial sum; a second single-block pass collapses the partials into the final scalar  
**Metric**: GB/s — reads 1 tensor → `(1 × n × bytes × 1e-9) / (ms × 1e-3)`

In [ ]:
# ── reduce_sum: Correctness ───────────────────────────────────────────────────
test_reduce_sum()

In [ ]:
# ── reduce_sum: Benchmark ─────────────────────────────────────────────────────
benchmark_reduce_sum.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/reductions",
)

**Interpretation**: Add notes here.

## 2. max_min

**File**: `kernels/reductions/max_min.py`  
**PyTorch equivalents**: `x.argmax()`, `x.argmin()`  
**Metric**: GB/s

In [ ]:
# ── max_min: Correctness ─────────────────────────────────────────────────────
# test_max_min()

In [ ]:
# ── max_min: Benchmark ───────────────────────────────────────────────────────
# benchmark_max_min.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/reductions/max_min_benchmark.png")

**Interpretation**: Add notes here.

## 3. softmax

**File**: `kernels/reductions/softmax.py`  
**PyTorch equivalent**: `torch.softmax(x, dim=-1)`  
**Key**: numerically stable via max subtraction (online softmax trick)  
**Metric**: GB/s — `(2 × n × bytes × 1e-9) / (ms × 1e-3)` (read + write)

In [ ]:
# ── softmax: Correctness ─────────────────────────────────────────────────────
# test_softmax()

In [ ]:
# ── softmax: Benchmark ───────────────────────────────────────────────────────
# benchmark_softmax.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/reductions/softmax_benchmark.png")

**Interpretation**: Add notes here.

## 4. layer_norm

**File**: `kernels/reductions/layer_norm.py`  
**PyTorch equivalent**: `torch.nn.functional.layer_norm(x, normalized_shape)`  
**Metric**: GB/s — `(2 × n × bytes × 1e-9) / (ms × 1e-3)`

In [ ]:
# ── layer_norm: Correctness ──────────────────────────────────────────────────
# test_layer_norm()

In [ ]:
# ── layer_norm: Benchmark ────────────────────────────────────────────────────
# benchmark_layer_norm.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/reductions/layer_norm_benchmark.png")

**Interpretation**: Add notes here.

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
# import pandas as pd, glob
# csvs = glob.glob("benchmarks/results/reductions/*.csv")
# if csvs:
#     print(pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True).to_string(index=False))
# else:
#     print("No CSVs yet.")